# Pipeline Colab PFE ML — Exécution complète

Ce notebook est la version finale d'exécution complète. Il évite les limites de test rapide et écrit des sorties durables sur Google Drive, tout en utilisant le disque local de Colab comme zone de transit rapide.

## Avant d'exécuter ce notebook partagé

Le propriétaire du notebook doit vous partager le dossier de données Google Drive avant toute exécution. Une fois l'accès accordé :

1. Ouvrez une fois le dossier Drive partagé dans Google Drive.
2. Ajoutez un raccourci vers votre propre Drive si le dossier n'est pas déjà sous `Mon Drive`.
3. Assurez-vous que le dossier est accessible depuis Colab à l'emplacement :

```text
/content/drive/MyDrive/PFE ML Data/pfe_data
```

4. Si votre dossier partagé apparaît ailleurs, modifiez `DRIVE_ROOT` dans la cellule de configuration avant de lancer le pipeline.
5. Si vous devez seulement consulter les sorties, un accès Lecteur suffit. Si vous devez exécuter le pipeline et écrire des résultats, il vous faut un accès Éditeur.
6. Les identifiants INPI ne sont pas stockés dans ce notebook. Si vous avez besoin de l'étape INPI, saisissez les identifiants de façon privée dans la cellule dédiée.

## 1. Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.1. Vérification du dossier Drive partagé

Après le montage de Drive, vérifiez que le dossier de données partagé du projet est bien visible. Un dossier Drive préparé doit à terme contenir :

```text
data-lake
ml-artifacts
reports
source-archives
```

Si le dossier est absent, ouvrez le dossier partagé dans Google Drive, ajoutez un raccourci vers `Mon Drive`, puis relancez cette vérification. Si le raccourci utilise un chemin différent, mettez à jour `DRIVE_ROOT` dans la cellule de configuration.

In [ ]:
EXPECTED_DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'

from pathlib import Path

REQUIRED_FOLDERS = ['data-lake', 'ml-artifacts', 'reports', 'source-archives']
OPTIONAL_DATA_LAKE_FOLDERS = ['raw', 'clean', 'features']

expected = Path(EXPECTED_DRIVE_ROOT)
print(f'Checking shared data folder: {expected}')

if not expected.exists():
    print(f'Creating root folder: {expected}')
    expected.mkdir(parents=True, exist_ok=True)

for folder in REQUIRED_FOLDERS:
    (expected / folder).mkdir(parents=True, exist_ok=True)
for folder in OPTIONAL_DATA_LAKE_FOLDERS:
    (expected / 'data-lake' / folder).mkdir(parents=True, exist_ok=True)
found = {child.name for child in expected.iterdir() if child.is_dir()}
missing = [name for name in REQUIRED_FOLDERS if name not in found]

print('Found top-level folders:')
for name in sorted(found):
    print(f'- {name}')

if missing:
    raise RuntimeError(
        'DRIVE CHECK FAILED: shared folder is visible, but the expected layout is incomplete. '
        f'Missing: {missing}. Expected: {REQUIRED_FOLDERS}'
    )

data_lake = expected / 'data-lake'
data_lake_found = {child.name for child in data_lake.iterdir() if child.is_dir()} if data_lake.exists() else set()
present_data_lake = [name for name in OPTIONAL_DATA_LAKE_FOLDERS if name in data_lake_found]

print('\nPASS: shared Drive folder is visible and has the expected top-level layout.')
print(f'Data-lake folders currently present: {present_data_lake or "none yet"}')
print('You can keep this path as DRIVE_ROOT.')

## 2. Clonage ou mise à jour du dépôt

In [ ]:
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

%cd /content
!if [ ! -d "$REPO_DIR/.git" ]; then git clone --branch data-extraction --single-branch https://github.com/zribi1/pfein.git "$REPO_DIR"; else cd "$REPO_DIR" && git pull; fi
%cd $BACKEND_DIR

## 3. Configuration des chemins

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
WORK_DIR = '/content/pfe_work'

!mkdir -p "$DRIVE_ROOT" "$WORK_DIR"
!df -h /content
!ls -lah "$DRIVE_ROOT"

## 4. Installation des dépendances Colab

In [ ]:
%cd $BACKEND_DIR
!pip install -q -r collabs/requirements-colab.txt

## 5. Identifiants INPI

N'exécutez cette cellule qu'après avoir remplacé les valeurs d'exemple. Ne laissez jamais d'identifiants dans Git ni dans des captures d'écran.

In [ ]:
import os

# Remplacez ces valeurs avant l'exécution finale INPI.
os.environ['INPI_FTP_HOST'] = 'your_host'
os.environ['INPI_FTP_PORT'] = '21'
os.environ['INPI_FTP_USER'] = 'your_user'
os.environ['INPI_FTP_PASSWORD'] = 'your_password'
os.environ['INPI_FTP_PROTOCOL'] = 'ftp'
os.environ['INPI_REMOTE_BASE_DIR'] = '/'

## 6. Pipeline de téléchargement / export des données

Exécutez une étape, attendez qu'elle se termine, puis remplacez `PIPELINE_STEP` par la valeur suivante et relancez la cellule. Chaque étape vérifie d'abord Drive ; si les artefacts attendus existent déjà, l'étape est ignorée au lieu de re-télécharger ou ré-exporter.

| Valeur | Rôle | Quand passer à la suite |
|---|---|---|
| `download_insee` | Télécharge les fichiers en masse INSEE Sirene vers Drive. | Le statut indique `download_insee` réussi ou ignoré car déjà fait. |
| `export_raw_insee` | Récupère les fichiers INSEE depuis Drive vers le stockage de travail local si besoin, puis exporte le Parquet brut INSEE. | Le statut indique `export_raw_insee` réussi ou ignoré. |
| `download_bilan` | Télécharge le Parquet public des bilans financiers vers Drive. | Le statut indique `download_bilan` réussi ou ignoré. |
| `export_raw_bilan` | Récupère le fichier financier depuis Drive si besoin, puis le copie vers le stockage brut. | Le statut indique `export_raw_bilan` réussi ou ignoré. |
| `download_inpi` | Télécharge les archives ZIP INPI vers Drive avec de courtes tentatives reprises. | Le statut indique `download_inpi` réussi ou ignoré. |
| `export_raw_inpi` | Récupère les ZIP INPI depuis Drive si besoin, puis exporte le Parquet brut INPI. | Le statut indique `export_raw_inpi` réussi ou ignoré. |
| `download_bodacc` | Télécharge les archives BODACC vers Drive pour les années demandées. | Le statut indique `download_bodacc` réussi ou ignoré. |
| `export_raw_bodacc` | Récupère les archives BODACC depuis Drive si besoin, puis exporte le Parquet brut BODACC. | Le statut indique `export_raw_bodacc` réussi ou ignoré. |
| `build_ml_data` | Construit les tables nettoyées, les labels, les features, l'entraînement optionnel et les rapports d'audit à partir de toutes les sources prêtes. | À lancer une fois que les sources dont vous avez besoin sont prêtes. |

In [ ]:
%cd $BACKEND_DIR

import os
import shlex
import subprocess
import sys
from pathlib import Path

PIPELINE_STEP = "download_insee"
FULL_DATA_RUN = False
MAX_COMPANIES = None if FULL_DATA_RUN else 100000
TRAIN_MODEL = False
RUN_AUDIT = not FULL_DATA_RUN
YEAR_BATCH_SIZE = 2 if FULL_DATA_RUN else None

DUCKDB_TMP = "/content/pfein_duckdb_tmp"
Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)
os.environ["DUCKDB_TEMP_DIRECTORY"] = DUCKDB_TMP
# Ordre recommandé :
# download_insee, export_raw_insee,
# download_bilan, export_raw_bilan,
# download_inpi, export_raw_inpi,
# download_bodacc, export_raw_bodacc,
# build_ml_data

command = [
    sys.executable,
    "-u",
    "collabs/full_pipeline.py",
    "--step",
    PIPELINE_STEP,
    "--drive-root",
    DRIVE_ROOT,
    "--work-dir",
    WORK_DIR,
    "--repo-dir",
    BACKEND_DIR,
    "--start-year",
    "2017",
    "--end-year",
    "2025",
]

if PIPELINE_STEP == "download_insee":
    command.append("--install-deps")
if PIPELINE_STEP == "download_inpi":
    command.extend(["--inpi-retries", "2"])
if PIPELINE_STEP in {"download_bodacc", "export_raw_bodacc"}:
    command.extend(["--bodacc-mode", "historical", "--bodacc-families", "PCL", "RCS-B"])
if PIPELINE_STEP == "build_ml_data":
    if TRAIN_MODEL:
        command.append("--train")
    if RUN_AUDIT:
        command.append("--audit")
    if MAX_COMPANIES is not None:
        command.extend(["--max-companies", str(MAX_COMPANIES)])
    if YEAR_BATCH_SIZE is not None:
        command.extend(["--year-batch-size", str(YEAR_BATCH_SIZE)])

print(" ".join(shlex.quote(part) for part in command))
subprocess.run(command, check=True)


## 7. Inspection des sorties finales

In [ ]:
!ls -lah "$DRIVE_ROOT/data-lake/features"
!ls -lah "$DRIVE_ROOT/ml-artifacts"
!ls -lah "$DRIVE_ROOT/reports"
!test -f "$DRIVE_ROOT/ml-artifacts/model_metadata.json" && cat "$DRIVE_ROOT/ml-artifacts/model_metadata.json" || echo "model_metadata.json not found yet"
!test -f "$DRIVE_ROOT/reports/data_lake_audit.md" && head -80 "$DRIVE_ROOT/reports/data_lake_audit.md" || echo "data_lake_audit.md not found yet"

## 7. Contrôle du statut du pipeline

À lancer après chaque étape du pipeline, surtout si Colab signale un dépassement de délai ou un échec partiel. Cette cellule lit le fichier de statut persistant depuis Drive et indique quelles étapes ont réussi, échoué ou été ignorées.

In [ ]:
import json
from pathlib import Path

status_json = Path(DRIVE_ROOT) / 'reports' / 'pipeline_status.json'
status_md = Path(DRIVE_ROOT) / 'reports' / 'pipeline_status.md'
print(f"Status JSON: {status_json}")
print(f"Status Markdown: {status_md}")

if not status_json.exists():
    print('No pipeline status file found yet. Run a pipeline step first.')
else:
    status = json.loads(status_json.read_text(encoding='utf-8'))
    for step in status.get('steps', []):
        name = step.get('step')
        state = step.get('status')
        error = step.get('error', '')
        print(f'- {name}: {state}' + (f' | {error}' if error else ''))
    failures = [s for s in status.get('steps', []) if s.get('status') == 'failed']
    if failures:
        print('\nFailures found. Open pipeline_status.md for tracebacks and rerun the failed step/resume command after fixing it.')
    else:
        print('\nNo failed steps recorded.')


## 8. Contrôles d'intégrité INPI

À lancer après les étapes de téléchargement / export INPI. Cette cellule vérifie si les téléchargements ZIP INPI sont terminés, s'il reste des fichiers `.part`, si les fichiers ZIP sont lisibles, et si les sorties Parquet INPI brutes / nettoyées existent.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, BadZipFile

drive_inpi = Path(DRIVE_ROOT) / 'source-archives' / 'inpi'
work_inpi = Path(WORK_DIR) / 'source-archives' / 'inpi'
roots = [path for path in [work_inpi, drive_inpi] if path.exists()]

print('INPI archive roots:')
for root in roots:
    print(f'- {root}')

zip_files = []
part_files = []
for root in roots:
    zip_files.extend(root.rglob('*.zip'))
    part_files.extend(root.rglob('*.part'))

print(f'ZIP files found: {len(zip_files)}')
for path in sorted(zip_files):
    print(f'- {path} ({path.stat().st_size / 1024 / 1024:,.1f} MB)')

if part_files:
    print('\nWARNING: incomplete INPI partial files remain:')
    for path in sorted(part_files):
        print(f'- {path} ({path.stat().st_size / 1024 / 1024:,.1f} MB)')
else:
    print('\nOK: no INPI .part files found.')

print('\nTesting ZIP readability:')
bad = []
for path in sorted(set(zip_files)):
    try:
        with ZipFile(path) as zf:
            bad_member = zf.testzip()
        if bad_member is None:
            print(f'OK: {path.name}')
        else:
            print(f'BAD MEMBER: {path.name} -> {bad_member}')
            bad.append(str(path))
    except BadZipFile as exc:
        print(f'BAD ZIP: {path.name} -> {exc}')
        bad.append(str(path))

raw_inpi = Path(DRIVE_ROOT) / 'data-lake' / 'raw' / 'inpi'
clean_accounts = Path(DRIVE_ROOT) / 'data-lake' / 'clean' / 'annual_accounts'
clean_formalities = Path(DRIVE_ROOT) / 'data-lake' / 'clean' / 'formalities_events'

print('\nParquet output checks:')
for label, root in [('raw_inpi', raw_inpi), ('clean_annual_accounts', clean_accounts), ('clean_formalities_events', clean_formalities)]:
    count = len(list(root.rglob('*.parquet'))) if root.exists() else 0
    print(f'{label}: {count} parquet file(s) under {root}')

if bad or part_files:
    raise RuntimeError('INPI integrity check failed: fix bad ZIP or resume incomplete .part downloads before trusting INPI outputs.')
print('\nPASS: INPI downloads are readable and no partial files remain.')

## 9. Audits de préparation ML

Génère l'audit des labels, l'audit de fuite de données et l'audit de sûreté des features. Ces rapports sont indispensables avant de présenter les métriques du modèle comme preuves finales.

In [ ]:
%cd $BACKEND_DIR

!python collabs/audit_ml_readiness.py \
  --drive-root "$DRIVE_ROOT" \
  --sample-rows 20

!ls -lah "$DRIVE_ROOT/reports/label_audit.md" "$DRIVE_ROOT/reports/leakage_audit.md" "$DRIVE_ROOT/reports/feature_safety_audit.md"
!head -80 "$DRIVE_ROOT/reports/label_audit.md"

## 10. Commandes de reprise

Si l'environnement d'exécution est réinitialisé alors que les téléchargements sont déjà sur Drive, relancez les cellules de configuration, puis relancez l'étape de pipeline en cours. Les scripts amorcent le répertoire de travail local depuis Drive et ignorent autant que possible les archives / manifestes lisibles déjà présents.